# AIC 2026 — Distributed DAM Object Description Worker (Kaggle)

Parameterized thin launcher for running `scripts/run_dam_batch.py` on a deterministic shard (`WORKER_ID` of `NUM_WORKERS`).

- Store any optional credentials in Kaggle Secrets (`RCLONE_CONFIG_GDRIVE`, `AIC_GITHUB_TOKEN`). Never paste secrets or personal paths into notebook cells.
- Replace the competition dataset paths below with your attached Kaggle Dataset slugs.

In [ ]:
from pathlib import Path
import os
import shutil
import subprocess

REPO_URL = "https://github.com/AIVIETNAM-AIO-Dewey/AIC-2026.git"
REPO_REF = "release/final"
SOURCE_MIRROR = Path("/kaggle/input/aic2026-source/AIC-2026")
TARGET_DIR = Path("/kaggle/working/AIC-2026")

if not TARGET_DIR.is_dir():
    if SOURCE_MIRROR.is_dir():
        shutil.copytree(SOURCE_MIRROR, TARGET_DIR)
    else:
        clone_env = os.environ.copy()
        try:
            from kaggle_secrets import UserSecretsClient
            token = UserSecretsClient().get_secret("AIC_GITHUB_TOKEN")
        except Exception:
            token = ""
        if token:
            clone_env.update({
                "GIT_CONFIG_COUNT": "1",
                "GIT_CONFIG_KEY_0": "http.https://github.com/.extraheader",
                "GIT_CONFIG_VALUE_0": f"Authorization: Bearer {token}",
            })
        subprocess.run(
            ["git", "clone", "--branch", REPO_REF, REPO_URL, str(TARGET_DIR)],
            env=clone_env,
            check=True,
        )
        del token, clone_env

os.chdir(TARGET_DIR)
print("Working directory:", Path.cwd())

Configure the worker shard (`WORKER_ID` in `0 .. NUM_WORKERS - 1`) and attached dataset roots.

In [ ]:
WORKER_ID = 0
NUM_WORKERS = 12

DATA_ROOT = "/kaggle/input/aic2026-data"
KEYFRAMES_ROOT = f"{DATA_ROOT}/keyframes"
OBJECTS_ROOT = f"{DATA_ROOT}/objects"
MAP_ROOT = f"{DATA_ROOT}/map-keyframes"
ARTIFACT_ROOT = "/kaggle/working/aic2026-artifacts"
CACHE_ROOT = "/kaggle/working/aic2026-model-cache"

# Optional remote sync target (requires RCLONE_CONFIG_GDRIVE Kaggle Secret). Set to "" to keep artifacts local.
RCLONE_DEST = ""

In [ ]:
%%bash
set -euo pipefail
python -m pip install --quiet --no-deps -r requirements/kaggle.txt

In [ ]:
import os
import re
import subprocess
from pathlib import Path

if RCLONE_DEST:
    from kaggle_secrets import UserSecretsClient

    subprocess.run(["apt-get", "update", "-qq"], check=True)
    subprocess.run(["apt-get", "install", "-y", "-qq", "rclone"], check=True)
    raw_secret = UserSecretsClient().get_secret("RCLONE_CONFIG_GDRIVE").strip()
    if not raw_secret.startswith("["):
        raw_secret = "[gdrive] " + raw_secret
    formatted = raw_secret
    if "\n" not in formatted:
        formatted = formatted.replace("[gdrive]", "[gdrive]\n")
        for key in ("type", "scope", "token", "client_id", "client_secret"):
            formatted = re.sub(rf"\s+({key}\s*=)", r"\n\1", formatted)
    rclone_conf = Path("/root/.config/rclone/rclone.conf")
    rclone_conf.parent.mkdir(parents=True, exist_ok=True)
    rclone_conf.write_text(formatted, encoding="utf-8")
    os.environ["RCLONE_CONFIG"] = str(rclone_conf)
    del raw_secret, formatted
    print("Configured rclone remotes:", subprocess.check_output(["rclone", "listremotes"], text=True).strip())
else:
    print("RCLONE_DEST is empty; skipping rclone setup.")

In [ ]:
import subprocess

dry_run_cmd = [
    "python",
    "scripts/run_dam_batch.py",
    "--worker-id", str(WORKER_ID),
    "--num-workers", str(NUM_WORKERS),
    "--keyframes-root", KEYFRAMES_ROOT,
    "--objects-root", OBJECTS_ROOT,
    "--map-keyframes-root", MAP_ROOT,
    "--dry-run",
]
if RCLONE_DEST:
    dry_run_cmd.extend(["--rclone-dest", RCLONE_DEST])
subprocess.run(dry_run_cmd, check=True)

In [ ]:
import subprocess

batch_cmd = [
    "python",
    "scripts/run_dam_batch.py",
    "--worker-id", str(WORKER_ID),
    "--num-workers", str(NUM_WORKERS),
    "--keyframes-root", KEYFRAMES_ROOT,
    "--objects-root", OBJECTS_ROOT,
    "--map-keyframes-root", MAP_ROOT,
    "--output-root", ARTIFACT_ROOT,
    "--cache-root", CACHE_ROOT,
    "--device", "cuda",
]
if RCLONE_DEST:
    batch_cmd.extend(["--rclone-dest", RCLONE_DEST])
subprocess.run(batch_cmd, check=True)